# Across-Plot Analysis: Average Sentence Order by Subcategory

**Goal:** For each article and each comment-tag subcategory, compute the
mean *order* — i.e., the average position within a comment where statements
of that category appear. This reveals whether certain reasoning types
(e.g., visual observations vs. inferences) tend to occur earlier or later
in a viewer's written response.

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ──────────────────────────────────────────────────────────────
CLASSIFICATIONS_DIR = Path("..") / "ace_classifications"
COMMENTS_DIR = Path("..") / "ace_comments"

# ── Load ace_classifications ───────────────────────────────────────────
cls_rows = []
for p in sorted(CLASSIFICATIONS_DIR.glob("*.json")):
    with p.open() as f:
        cls_rows.extend(json.load(f))

df_cls = pd.DataFrame(cls_rows)
df_cls["article_id"] = df_cls["article_id"].astype(str)
df_cls["comment_id"] = df_cls["comment_id"].astype(int)

# ── Load ace_comments and flatten the order dict ───────────────────────
# Each ace_comments file is a JSON object with an `order` dict mapping
# sentence text → position number within that comment.
order_rows = []
for p in sorted(COMMENTS_DIR.glob("*/*.json")):
    with p.open() as f:
        doc = json.load(f)
    art_id = str(doc["article_id"])
    cmt_idx = int(doc["comment_index"])
    order_map = doc.get("order", {})
    if order_map:
        for sentence, pos in order_map.items():
            order_rows.append(
                {"article_id": art_id, "comment_id": cmt_idx,
                 "original_comment": sentence, "order": pos}
            )
    else:
        # Fallback: derive order from position in ace_sentences list
        for i, sentence in enumerate(doc.get("ace_sentences", []), start=1):
            order_rows.append(
                {"article_id": art_id, "comment_id": cmt_idx,
                 "original_comment": sentence, "order": i}
            )

df_order = pd.DataFrame(order_rows)

print(f"Classifications: {len(df_cls):,} rows, {df_cls['article_id'].nunique()} articles")
print(f"Order records:   {len(df_order):,} rows")

Classifications: 530,063 rows, 191 articles
Order records:   544,640 rows


In [21]:
# ── Merge classifications with order info ──────────────────────────────
df_merged = pd.merge(
    df_cls,
    df_order,
    on=["article_id", "comment_id", "original_comment"],
    how="left",
)

print(f"Merged rows: {len(df_merged):,}")
print(f"Order available: {df_merged['order'].notna().sum():,} / {len(df_merged):,}")

# ── Normalise tags ─────────────────────────────────────────────────────
_TAG_CLEANUP = {
    "L3: Trend and pattern analysis": "Visual Observation: Cross-point Pattern Recognition",
    "L1: Elemental and encoded properties": "Visual Observation: Chart Structure & Text",
    "L2: Statistical concepts and relations": "Visual Observation: Data Point Extraction",
    "VO1: Chart Structure, Layout & Text": "Visual Observation: Chart Structure & Text",
    "VO2: Data Point Reading": "Visual Observation: Data Point Extraction",
    "VO3: Comparisons, Trends & Patterns": "Visual Observation: Cross-point Pattern Recognition",
    "Background knowledge": "Prior Knowledge: Background",
    "Personal/episodic retrieval": "Prior Knowledge: Personal / Episodic",
    "Prior Knowledge: Personal /Episodic": "Prior Knowledge: Personal / Episodic",
    "Evaluative / affective judgment": "Evaluative: Reactive",
    "Explanatory inference": "Inference: Explanatory",
    "Predictive / counterfactual inference": "Inference: Predictive / Hypothetical",
    "Information need / curiosity": "Curiosity",
    "Meta /Paratext": "Meta / Paratext",
    "Meta / paratext": "Meta / Paratext",
}
df_merged["comment_tag"] = df_merged["comment_tag"].replace(_TAG_CLEANUP)

# ── Drop excluded categories ──────────────────────────────────────────
EXCLUDED = {"Meta / Paratext", "Uncategorizable"}
df_merged = df_merged[~df_merged["comment_tag"].isin(EXCLUDED)].copy()

print(f"\nAfter filtering: {len(df_merged):,} rows, "
      f"{df_merged['article_id'].nunique()} articles")

Merged rows: 530,091
Order available: 529,807 / 530,091

After filtering: 448,007 rows, 191 articles


In [22]:
# ── Average order per article × subcategory ────────────────────────────
avg_order_by_article = (
    df_merged
    .groupby(["article_id", "comment_tag"])["order"]
    .mean()
    .unstack()
)

with pd.option_context("display.max_columns", None, "display.width", 140,
                        "display.float_format", "{:.2f}".format):
    display(avg_order_by_article)

comment_tag,Curiosity,Evaluative: Prescriptive,Evaluative: Reactive,Inference: Explanatory,Inference: Predictive / Hypothetical,Prior Knowledge: Background,Prior Knowledge: Personal / Episodic,Visual Observation: Chart Structure & Text,Visual Observation: Cross-point Pattern Recognition,Visual Observation: Data Point Extraction
article_id,,,,,,,,,,
1,8.02,12.30,11.69,10.38,11.25,11.15,10.26,4.32,5.13,5.45
10,6.34,12.32,9.49,9.81,13.08,11.23,9.16,3.91,3.81,3.76
100,8.42,14.63,10.27,11.70,12.57,11.54,9.85,6.90,7.87,6.30
101,6.33,8.29,6.48,8.13,7.34,8.70,9.74,6.06,3.84,3.64
102,8.33,8.89,10.08,8.89,11.09,9.85,10.94,8.81,5.76,6.22
...,...,...,...,...,...,...,...,...,...,...
95,10.66,12.47,11.13,11.82,11.91,12.77,12.48,9.50,8.38,11.30
96,7.09,17.29,10.57,8.49,10.57,9.97,9.21,7.03,5.14,5.83
97,7.34,12.33,11.82,9.52,11.70,10.84,10.05,8.24,5.46,9.29


In [23]:

# Connected dot plot (slopegraph-like) - one dot per article for each subcategory, connected per subcategory

import altair as alt

# Long format: already in avg_long from previous cell
# Ensure the article_id sorting
avg_long_sorted = avg_long.sort_values(["article_id", "comment_tag"])

# Fixed order and explicit color mapping for subcategories, matching prior plot scheme:
subcategory_order = [
    "Visual Observation: Chart Structure & Text",
    "Visual Observation: Data Point Extraction",
    "Visual Observation: Cross-point Pattern Recognition",
    "Prior Knowledge: Background",
    "Prior Knowledge: Personal / Episodic",
    "Evaluative: Prescriptive",
    "Evaluative: Reactive",
    "Inference: Explanatory",
    "Inference: Predictive / Hypothetical",
    "Curiosity",
]

# Color scheme for subcategories (reusing color scheme as in this notebook)
SUBCATEGORY_COLORS = [
    "#6666CC",   # Visual Observation: Chart Structure & Text (blue)
    "#1696D2",   # Visual Observation: Data Point Extraction (cyan/blue)
    "#7C1E6A",   # Visual Observation: Cross-point Pattern Recognition (violet)
    "#FDBF11",   # Prior Knowledge: Background (yellow)
    "#E17C05",   # Prior Knowledge: Personal / Episodic (orange)
    "#9C964A",   # Evaluative: Prescriptive (olive)
    "#DC3977",   # Evaluative: Reactive (magenta/pink)
    "#129B7D",   # Inference: Explanatory (green)
    "#272B4D",   # Inference: Predictive / Hypothetical (dark blue/indigo)
    "#B2B2B2",   # Curiosity (gray)
]

color_scale = alt.Scale(
    domain=subcategory_order,
    range=SUBCATEGORY_COLORS
)

# We'll plot lines connecting subcategories for each article,
# and overlay dots for each point

base = alt.Chart(avg_long_sorted).encode(
    x=alt.X('article_id:N', title='Article ID', sort=alt.EncodingSortField(field="article_id", order="ascending")),
    y=alt.Y('avg_order:Q', title='Average Sentence Order', scale=alt.Scale(zero=False)),
    color=alt.Color('comment_tag:N',
                   title="Comment Tag",
                   sort=subcategory_order,
                   scale=color_scale),
    detail='comment_tag:N',
    tooltip=[
        alt.Tooltip("article_id:N"),
        alt.Tooltip("comment_tag:N"),
        alt.Tooltip("avg_order:Q", format=".2f")
    ]
)

lines = base.mark_line(point=False, interpolate='monotone')
dots = base.mark_circle(size=55, opacity=1)

chart = alt.layer(
    lines,
    dots
).properties(
    title="Connected Dot Plot: Average Sentence Order per Article and Subcategory",
    width=1500,
    height=420
).configure_axisX(
    labelAngle=90
)

chart


alt.LayerChart(...)

In [25]:
# ── Grand mean order across all articles ───────────────────────────────
grand_mean = (
    df_merged
    .groupby("comment_tag")["order"]
    .mean()
    .sort_values()
)
print("Grand mean order (across all articles):\n")
print(grand_mean.to_string(float_format="{:.2f}".format))

Grand mean order (across all articles):

comment_tag
Visual Observation: Cross-point Pattern Recognition    4.48
Visual Observation: Data Point Extraction              4.99
Visual Observation: Chart Structure & Text             5.68
Curiosity                                              6.50
Inference: Explanatory                                 8.21
Evaluative: Reactive                                   8.43
Prior Knowledge: Background                            9.07
Prior Knowledge: Personal / Episodic                   9.27
Inference: Predictive / Hypothetical                   9.55
Evaluative: Prescriptive                              10.67


In [26]:
# Layered scatter plot: x=subcategories (ordered by grand mean), y=mean order

import numpy as np

# 1. Compute per-(article, category) means, used for scattered (jittered) small dots
per_article_subcat = (
    df_merged.groupby(['article_id', 'comment_tag'])['order']
    .mean()
    .reset_index()
)

# 2. Compute the grand mean by subcategory (all comments), used for large dots & sorting
grand_subcat_means = (
    df_merged.groupby('comment_tag')['order']
    .mean()
    .reset_index()
    .rename(columns={'order': 'grand_mean_order'})
)

# Sort subcategories by ascending grand mean (lowest mean order to highest)
cat_order = list(grand_subcat_means.sort_values('grand_mean_order')['comment_tag'])

# Jitter x for small dots (using order in sorted list)
np.random.seed(0)
cat_to_x = {cat: i for i, cat in enumerate(cat_order)}
per_article_subcat['jitter'] = (
    per_article_subcat['comment_tag']
    .map(cat_to_x)
    + np.random.uniform(-0.18, 0.18, size=len(per_article_subcat))
)

## Large dots for grand means (now sorted)
large_dots = alt.Chart(grand_subcat_means).mark_circle(size=220, color="black").encode(
    x=alt.X('comment_tag:N', title='Subcategory', sort=cat_order),
    y=alt.Y('grand_mean_order:Q', title='Mean Order (all comments)', scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("comment_tag:N"),
        alt.Tooltip("grand_mean_order:Q", format=".2f")
    ]
)

## Scatter small dots for per-article means (w/ jitter on x, sorted categories)
small_dots = alt.Chart(per_article_subcat).mark_circle(size=55, color="steelblue", opacity=0.45).encode(
    x=alt.X('jitter:Q',
        title='Subcategory',
        axis=alt.Axis(labels=False, ticks=False, domain=False),
        scale=alt.Scale(domain=[-0.5, len(cat_order)-0.5]),
    ),
    y=alt.Y('order:Q', title='Mean Order in Article', scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip("comment_tag:N"),
        alt.Tooltip("article_id:N"),
        alt.Tooltip("order:Q", format=".2f")
    ]
)

# Invisible overlay to show x-axis ticks/labels at correct locations
x_labels = alt.Chart(grand_subcat_means).mark_text(opacity=0).encode(
    x=alt.X('comment_tag:N', title='Subcategory', sort=cat_order, axis=alt.Axis(labelAngle=36)),
    y=alt.value(0)
)

chart = alt.layer(
    small_dots,
    large_dots,
    x_labels
).properties(
    title="Mean Sentence Order: Grand Subcategory Means and Distributions Across Articles",
    width=600,
    height=430
).configure_axis(
    labelFontSize=13,
    titleFontSize=15
)

chart

alt.LayerChart(...)